# Baseline: CoLES via PyTorch-LifeStream

Prefix-safe CoLES baseline for low-label comparison against `kaggle_forecast_pipeline.ipynb`. The notebook uses deterministic forecast-style prefix cuts, train-prefix preprocessing, PTLS `ColesDataset`/`SampleSlices` when available, and standardized low-label result artifacts.


In [ ]:
# Cell 1 - Setup
import copy
import hashlib
import importlib.util
import json
import pathlib
import random
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")


def pip_install(*packages):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", *packages, "--quiet"],
        check=True,
    )


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        "/kaggle/working/denoising-event-sequences",
        "--quiet",
    ],
    check=True,
)
sys.path.insert(0, "/kaggle/working/denoising-event-sequences")

required_packages = {"pytorch-lifestream": "ptls"}
missing_packages = [
    pkg for pkg, module_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    pip_install(*missing_packages)


def check_numpy_stack():
    return subprocess.run(
        [
            sys.executable,
            "-c",
            "import numpy; import numpy.random; import pandas; import sklearn; import scipy",
        ],
        text=True,
        capture_output=True,
    )


stack_check = check_numpy_stack()
if stack_check.returncode != 0:
    print("Repairing NumPy/pandas/scikit-learn/scipy binary stack...")
    print(stack_check.stderr[-2000:])
    pip_install(
        "--force-reinstall",
        "--no-cache-dir",
        "numpy==1.26.4",
        "pandas==2.2.3",
        "scikit-learn==1.5.2",
        "scipy==1.13.1",
    )
    stack_check = check_numpy_stack()
    if stack_check.returncode != 0:
        raise RuntimeError(
            "Scientific Python stack is still binary-incompatible after reinstall. "
            "Restart the Kaggle session and run the notebook from the first cell again.\n"
            + stack_check.stderr[-2000:]
        )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader

from ptls.data_load.utils import collate_feature_dict
from ptls.nn import RnnSeqEncoder, TrxEncoder

try:
    from ptls.frames.coles import ColesDataset
    from ptls.frames.coles.split_strategy import SampleSlices
    PTLS_COLES_DATASET_AVAILABLE = True
except Exception as exc:
    # Some Kaggle images have Lightning/NumPy ABI issues. Keep the training loop
    # independent from Lightning and fall back to the same multi-slice objective.
    PTLS_COLES_DATASET_AVAILABLE = False
    ColesDataset = None
    SampleSlices = None
    print("PTLS ColesDataset unavailable; using manual multi-slice fallback:", repr(exc))

from src.data.splits import make_entity_splits
from src.utils.seed import set_seed

SMOKE_RUN = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print("Device:", device)


In [ ]:
# Cell 2 - Paths and config
WORKING_DIR = pathlib.Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
OUTPUT_DIR = WORKING_DIR / "outputs" / "baselines" / "coles_ptls"
CKPT_DIR = OUTPUT_DIR / "checkpoints"
for p in [OUTPUT_DIR, CKPT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

config = {
    "data": {
        "max_seq_len": 256,
        "min_seq_len": 2,
        "event_type_col": "MCC",
        "timestamp_col": "TRDATETIME",
        "target_col": "target_flag",
        "numerical_cols": ["amount"],
        "categorical_cols": ["channel_type", "trx_category"],
        "group_col": "cl_id",
        "truncation_pretrain": "sliding_window",
        "truncation_eval": "last_events",
        "amount_transform": "robust_scaler",
        "time_transform": "none",
        "train_ratio": 0.70,
        "val_ratio": 0.15,
        "test_ratio": 0.15,
        "min_vocab_count": 5,
    },
    "model": {
        "event_type_emb_dim": 64,
        "cat_emb_dim": 32,
        "num_projection_dim": 64,
        "time_projection_dim": 64,
        "hidden_dim": 256,
        "num_layers": 4,
        "num_heads": 8,
        "dim_feedforward": 1024,
        "dropout": 0.10,
        "activation": "gelu",
        "max_seq_len": 256,
    },
    "pooling": {"type": "gated_attention"},
    "forecasting": {
        "cut_min_ratio": 0.50,
        "cut_max_ratio": 0.80,
        "min_future_events": 2,
        "cut_seed": 42,
    },
    "training": {
        "batch_size": 128,
        "num_epochs_finetune": 20,
        "lr": 3e-4,
        "lr_encoder": 3e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "gradient_clip_val": 1.0,
        "mixed_precision": USE_AMP,
        "early_stopping_patience": 5,
        "log_every_n_steps": 50,
    },
}

LABEL_FRACTIONS = [1.00] if SMOKE_RUN else [0.05, 0.25, 0.50, 0.75, 1.00]
LABEL_SAMPLING_SEEDS = [42] if SMOKE_RUN else [42, 43, 44, 45, 46]
METRIC_COLS = ["accuracy", "macro_f1", "weighted_f1", "roc_auc", "pr_auc", "balanced_accuracy"]

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
A100_TUNED = "A100" in GPU_NAME.upper()
EMBEDDING_SIZE = 256
COLES_EPOCHS = 1 if SMOKE_RUN else 15
COLES_FINETUNE_EPOCHS = 2 if SMOKE_RUN else 20
COLES_SPLIT_COUNT = 2 if SMOKE_RUN else 5
COLES_BATCH_SIZE = 32 if SMOKE_RUN else (1024 if A100_TUNED else 256 if device.type == "cuda" else 64)
COLES_NUM_WORKERS = 0 if SMOKE_RUN else (4 if device.type == "cuda" else 0)
COLES_HEAD_LR = 3e-4
COLES_ENCODER_LR = 3e-5
COLES_PRETRAIN_LR = 1e-3
COLES_WEIGHT_DECAY = 1e-4
COLES_PATIENCE = 2 if SMOKE_RUN else 5
COLES_DOWNSTREAM_MODES = ["frozen_head", "full_finetune"]
PIN_MEMORY = device.type == "cuda"
DATALOADER_KWARGS = {"num_workers": COLES_NUM_WORKERS, "pin_memory": PIN_MEMORY}
if COLES_NUM_WORKERS > 0:
    DATALOADER_KWARGS["persistent_workers"] = True
    DATALOADER_KWARGS["prefetch_factor"] = 4

print("Output dir:", OUTPUT_DIR)
print("GPU:", GPU_NAME)
print("CoLES batch size:", COLES_BATCH_SIZE)
print("PTLS ColesDataset available:", PTLS_COLES_DATASET_AVAILABLE)
print("Cutoff policy:", config["forecasting"])


In [ ]:
# Cell 3 - Load data, shared splits, and prefix-only events
GROUP_COL = config["data"]["group_col"]
TARGET_COL = config["data"]["target_col"]
TIME_COL = config["data"]["timestamp_col"]
EVENT_COL = config["data"]["event_type_col"]
NUM_COL = "amount"
CAT_COLS = config["data"]["categorical_cols"]

def _forecast_cut_bounds(n_events, cfg):
    f_cfg = cfg.get("forecasting", {})
    min_ratio = float(f_cfg.get("cut_min_ratio", 0.50))
    max_ratio = float(f_cfg.get("cut_max_ratio", 0.80))
    min_future = int(f_cfg.get("min_future_events", 2))
    if n_events < min_future + 1:
        return None
    cut_low = max(1, int(np.floor(n_events * min_ratio)))
    cut_high = min(n_events - min_future, int(np.floor(n_events * max_ratio)))
    if cut_high < cut_low:
        cut_low = max(1, n_events - min_future)
        cut_high = n_events - min_future
    if cut_high < cut_low:
        return None
    return cut_low, cut_high


def _stable_entity_seed(entity_id, base_seed=42):
    digest = hashlib.blake2b(f"{base_seed}:{entity_id}".encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, byteorder="little", signed=False)


def deterministic_forecast_cut(entity_id, n_events, cfg, base_seed=None):
    bounds = _forecast_cut_bounds(int(n_events), cfg)
    if bounds is None:
        return None
    cut_low, cut_high = bounds
    if base_seed is None:
        base_seed = int(cfg.get("forecasting", {}).get("cut_seed", 42))
    rng = np.random.default_rng(_stable_entity_seed(entity_id, base_seed))
    return int(rng.integers(cut_low, cut_high + 1))


def build_prefix_events(df_source, entity_ids, cfg):
    entity_set = set(entity_ids)
    working = df_source[df_source[GROUP_COL].isin(entity_set)].copy()
    working = working.sort_values([GROUP_COL, TIME_COL], kind="stable").reset_index(drop=True)
    working["forecast_event_pos"] = working.groupby(GROUP_COL, sort=False).cumcount()
    counts = working.groupby(GROUP_COL, sort=False).size()
    cut_by_entity = {}
    for entity_id in entity_ids:
        if entity_id not in counts.index:
            continue
        cut = deterministic_forecast_cut(entity_id, int(counts.loc[entity_id]), cfg)
        if cut is not None:
            cut_by_entity[entity_id] = cut
    working["forecast_cut"] = working[GROUP_COL].map(cut_by_entity)
    prefix = working[
        working["forecast_cut"].notna()
        & (working["forecast_event_pos"] < working["forecast_cut"])
    ].copy()
    prefix["forecast_cut"] = prefix["forecast_cut"].astype(int)
    valid_entity_ids = [entity_id for entity_id in entity_ids if entity_id in cut_by_entity]
    return prefix.reset_index(drop=True), cut_by_entity, valid_entity_ids


def filter_splits_to_valid(splits_in, valid_entity_ids):
    valid = set(valid_entity_ids)
    filtered = {name: [entity_id for entity_id in ids if entity_id in valid] for name, ids in splits_in.items()}
    if any(len(ids) == 0 for ids in filtered.values()):
        raise ValueError(f"Empty split after prefix filtering: { {k: len(v) for k, v in filtered.items()} }")
    return filtered


def assert_no_future_leakage(prefix_df, cut_by_entity):
    counts = prefix_df.groupby(GROUP_COL, sort=False).size()
    for entity_id, cut in cut_by_entity.items():
        if entity_id not in counts.index:
            continue
        observed = int(counts.loc[entity_id])
        if observed != int(cut):
            raise AssertionError(f"Prefix length mismatch for {entity_id}: {observed} != cut {cut}")
    max_pos = prefix_df.groupby(GROUP_COL, sort=False)["forecast_event_pos"].max()
    bad = [entity_id for entity_id, pos in max_pos.items() if int(pos) >= int(cut_by_entity[entity_id])]
    if bad:
        raise AssertionError(f"Future events leaked into prefix for entities: {bad[:5]}")


df = pd.read_csv(DATA_DIR / "rosbank" / "train.csv.gz")
df[TIME_COL] = pd.to_datetime(df[TIME_COL], format="%d%b%y:%H:%M:%S")
labels_by_entity = df.groupby(GROUP_COL)[TARGET_COL].first().to_dict()

raw_splits = make_entity_splits(
    df,
    entity_col=GROUP_COL,
    target_col=TARGET_COL,
    train_ratio=config["data"]["train_ratio"],
    val_ratio=config["data"]["val_ratio"],
    test_ratio=config["data"]["test_ratio"],
    seed=42,
)
raw_entity_ids = raw_splits["train"] + raw_splits["val"] + raw_splits["test"]
df_prefix, forecast_cut_by_entity, valid_entity_ids = build_prefix_events(df, raw_entity_ids, config)
splits = filter_splits_to_valid(raw_splits, valid_entity_ids)
all_entity_ids = splits["train"] + splits["val"] + splits["test"]
df_prefix = df_prefix[df_prefix[GROUP_COL].isin(set(all_entity_ids))].copy()
forecast_cut_by_entity = {entity_id: forecast_cut_by_entity[entity_id] for entity_id in all_entity_ids}
assert_no_future_leakage(df_prefix, forecast_cut_by_entity)

print("Raw events:", df.shape)
print("Prefix events:", df_prefix.shape)
print("Split sizes:", {k: len(v) for k, v in splits.items()})
print("Filtered short entities:", len(raw_entity_ids) - len(all_entity_ids))


In [ ]:
# Cell 4 - Shared sampling, metrics, aggregation, and plotting helpers
def get_entity_labels(df_source, group_col, target_col):
    return df_source.groupby(group_col)[target_col].first().to_dict()


def sample_labeled_entities(train_ids, labels_by_entity, fraction, seed):
    train_ids = list(train_ids)
    if fraction >= 1.0:
        return train_ids

    rng = np.random.default_rng(seed)
    by_label = {}
    for entity_id in train_ids:
        label = int(labels_by_entity[entity_id])
        by_label.setdefault(label, []).append(entity_id)

    sampled = []
    for label, ids in sorted(by_label.items()):
        ids = np.array(ids, dtype=object)
        n = max(1, int(round(len(ids) * fraction)))
        n = min(n, len(ids))
        sampled.extend(rng.choice(ids, size=n, replace=False).tolist())

    rng.shuffle(sampled)
    return sampled


def compute_binary_metrics(y_true, pos_proba):
    y_true = np.asarray(y_true).astype(int)
    pos_proba = np.asarray(pos_proba).astype(float)
    y_pred = (pos_proba >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, pos_proba)),
        "pr_auc": float(average_precision_score(y_true, pos_proba)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def json_default(obj):
    if isinstance(obj, pathlib.Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        value = float(obj)
        return None if not np.isfinite(value) else value
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if pd.isna(obj):
        return None
    return str(obj)


def aggregate_low_label_runs(run_rows, metric_cols=METRIC_COLS):
    runs_df = pd.DataFrame(run_rows)
    if runs_df.empty:
        return runs_df, []
    agg_rows = []
    for (pipeline, fraction), group in runs_df.groupby(["pipeline", "fraction"], sort=True):
        row = {"pipeline": pipeline, "fraction": float(fraction), "n_seeds": int(group["seed"].nunique())}
        for metric in metric_cols:
            values = pd.to_numeric(group[metric], errors="coerce").dropna()
            if len(values):
                row[f"{metric}_mean"] = float(values.mean())
                row[f"{metric}_std"] = float(values.std(ddof=0))
        agg_rows.append(row)
    agg_df = pd.DataFrame(agg_rows).sort_values(["fraction", "pipeline"]).reset_index(drop=True)
    return agg_df, agg_rows


def build_summary_table(agg_rows):
    rows = []
    for row in agg_rows:
        rows.append(
            {
                "pipeline": row["pipeline"],
                "fraction": row["fraction"],
                "n_seeds": row["n_seeds"],
                "roc_auc": f"{row.get('roc_auc_mean', float('nan')):.4f} +/- {row.get('roc_auc_std', 0.0):.4f}",
                "pr_auc": f"{row.get('pr_auc_mean', float('nan')):.4f} +/- {row.get('pr_auc_std', 0.0):.4f}",
                "macro_f1": f"{row.get('macro_f1_mean', float('nan')):.4f} +/- {row.get('macro_f1_std', 0.0):.4f}",
                "balanced_accuracy": f"{row.get('balanced_accuracy_mean', float('nan')):.4f} +/- {row.get('balanced_accuracy_std', 0.0):.4f}",
            }
        )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["fraction", "pipeline"]).reset_index(drop=True)


def plot_low_label_metrics(agg_df, output_dir, metrics=("roc_auc", "pr_auc", "macro_f1")):
    plot_dir = output_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    plot_paths = {}
    if agg_df.empty:
        return plot_paths
    for metric in metrics:
        mean_col = f"{metric}_mean"
        std_col = f"{metric}_std"
        if mean_col not in agg_df.columns:
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        for pipeline, group in agg_df.groupby("pipeline", sort=True):
            group = group.sort_values("fraction")
            x = pd.to_numeric(group["fraction"], errors="coerce")
            y = pd.to_numeric(group[mean_col], errors="coerce")
            yerr = pd.to_numeric(group.get(std_col, 0.0), errors="coerce").fillna(0.0)
            ax.errorbar(x, y, yerr=yerr, marker="o", linewidth=2, capsize=4, label=pipeline)
        ax.set_title(f"{metric} by label fraction")
        ax.set_xlabel("Label fraction")
        ax.set_ylabel(metric)
        ax.set_ylim(0.0, 1.0)
        ax.grid(True, alpha=0.25)
        ax.legend()
        path = plot_dir / f"{metric}_by_fraction.png"
        fig.savefig(path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        plot_paths[metric] = path
    return plot_paths


def save_low_label_outputs(run_rows, prediction_rows, baseline_name, output_dir, extra_artifacts=None):
    output_dir.mkdir(parents=True, exist_ok=True)
    all_runs_df = pd.DataFrame(run_rows)
    if not all_runs_df.empty:
        all_runs_df = all_runs_df.sort_values(["pipeline", "fraction", "seed"]).reset_index(drop=True)
    all_runs_path = output_dir / "low_label_all_runs.csv"
    all_runs_df.to_csv(all_runs_path, index=False)

    agg_df, agg_rows = aggregate_low_label_runs(run_rows)
    agg_path = output_dir / "low_label_aggregate.csv"
    agg_json_path = output_dir / "low_label_aggregate.json"
    agg_df.to_csv(agg_path, index=False)
    agg_json_path.write_text(json.dumps(agg_rows, indent=2, ensure_ascii=False, default=json_default))

    summary_df = build_summary_table(agg_rows)
    summary_path = output_dir / "low_label_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    predictions_df = pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame()
    pred_path = output_dir / "test_predictions.csv"
    predictions_df.to_csv(pred_path, index=False)

    plot_paths = plot_low_label_metrics(agg_df, output_dir)
    metrics_payload = {
        "baseline": baseline_name,
        "cutoff_policy": config.get("forecasting", {}),
        "metric_columns": METRIC_COLS,
        "split_sizes": {k: len(v) for k, v in splits.items()},
        "artifacts": {
            "low_label_all_runs": all_runs_path,
            "low_label_aggregate": agg_path,
            "low_label_aggregate_json": agg_json_path,
            "low_label_summary": summary_path,
            "test_predictions": pred_path,
            "plots": plot_paths,
            **(extra_artifacts or {}),
        },
        "aggregate": agg_rows,
    }
    metrics_path = output_dir / "metrics.json"
    metrics_path.write_text(json.dumps(metrics_payload, indent=2, ensure_ascii=False, default=json_default))

    print("Low-label results saved:")
    for path in [all_runs_path, agg_path, agg_json_path, summary_path, pred_path, metrics_path]:
        print(" -", path)
    for path in plot_paths.values():
        print(" -", path)
    print("\nLow-label summary:")
    print(summary_df.to_string(index=False))
    try:
        display(summary_df)
    except NameError:
        pass
    return all_runs_df, agg_df, summary_df, predictions_df, metrics_payload


In [ ]:
# Cell 5 - PTLS preprocessing fitted on train-prefix only
def fit_category_maps(df_train_prefix, cat_cols):
    maps = {}
    for col in cat_cols:
        values = df_train_prefix[col].astype(str).value_counts().index.tolist()
        mapping = {"<UNK>": 1}
        for value in values:
            if value not in mapping:
                mapping[value] = len(mapping) + 1
        maps[col] = mapping
    return maps


def encode_categories(df_source, maps):
    out = df_source.copy()
    for col, mapping in maps.items():
        out[col] = out[col].astype(str).map(lambda x, m=mapping: m.get(x, 1)).astype("int64")
    return out


def fit_amount_scaler(df_train_prefix):
    values = pd.to_numeric(df_train_prefix[NUM_COL], errors="coerce").fillna(0.0).values.astype(float)
    center = float(np.median(values))
    q75, q25 = np.percentile(values, [75, 25])
    scale = float(q75 - q25) or 1.0
    return center, scale


def make_ptls_records(df_encoded, entity_ids, include_target=False):
    records = []
    entity_set = set(entity_ids)
    subset = df_encoded[df_encoded[GROUP_COL].isin(entity_set)].sort_values([GROUP_COL, TIME_COL], kind="stable")
    for entity_id, ent in subset.groupby(GROUP_COL, sort=False):
        rec = {
            "event_time": torch.tensor(ent["event_time"].values, dtype=torch.float32),
            EVENT_COL: torch.tensor(ent[EVENT_COL].values, dtype=torch.long),
            "channel_type": torch.tensor(ent["channel_type"].values, dtype=torch.long),
            "trx_category": torch.tensor(ent["trx_category"].values, dtype=torch.long),
            NUM_COL: torch.tensor(ent[NUM_COL].values, dtype=torch.float32),
        }
        if include_target:
            rec["target"] = torch.tensor(int(labels_by_entity[entity_id]), dtype=torch.long)
        records.append((entity_id, rec))
    by_id = {entity_id: rec for entity_id, rec in records}
    return [by_id[entity_id] for entity_id in entity_ids if entity_id in by_id]

train_df = df_prefix[df_prefix[GROUP_COL].isin(set(splits["train"]))]
category_maps = fit_category_maps(train_df, [EVENT_COL] + CAT_COLS)
amount_center, amount_scale = fit_amount_scaler(train_df)

df_ptls = encode_categories(df_prefix, category_maps)
df_ptls[NUM_COL] = pd.to_numeric(df_ptls[NUM_COL], errors="coerce").fillna(amount_center)
df_ptls[NUM_COL] = (df_ptls[NUM_COL] - amount_center) / amount_scale
df_ptls["event_time"] = df_ptls[TIME_COL].astype("int64") // 10**9

train_records = make_ptls_records(df_ptls, splits["train"])
val_records = make_ptls_records(df_ptls, splits["val"])
test_records = make_ptls_records(df_ptls, splits["test"])
all_records = make_ptls_records(df_ptls, all_entity_ids)

embedding_specs = {
    col: {"in": max(mapping.values()) + 1, "out": 32 if col == EVENT_COL else 16}
    for col, mapping in category_maps.items()
}
print("PTLS records:", len(train_records), len(val_records), len(test_records))
print("Embedding specs:", embedding_specs)


In [ ]:
# Cell 6 - CoLES-style multi-slice contrastive pretraining
MIN_SLICE_LEN = 10
CONTRASTIVE_TEMPERATURE = 0.10
GRAD_CLIP = 1.0


def seed_all(seed):
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def slice_record(record, start, end):
    return {k: v[start:end] for k, v in record.items()}


def sample_slice(record):
    n = int(record["event_time"].shape[0])
    if n <= 1:
        return slice_record(record, 0, n)
    min_len = min(MIN_SLICE_LEN, n)
    max_len = min(config["data"]["max_seq_len"], n)
    if min_len > max_len:
        min_len = max_len
    length = random.randint(min_len, max_len)
    start = random.randint(0, n - length)
    return slice_record(record, start, start + length)


def collate_coles_views(records):
    views = []
    pair_ids = []
    for pair_id, record in enumerate(records):
        for _ in range(COLES_SPLIT_COUNT):
            views.append(sample_slice(record))
            pair_ids.append(pair_id)
    batch = collate_feature_dict(views)
    batch.payload["pair_id"] = torch.tensor(pair_ids, dtype=torch.long)
    return batch


def make_manual_coles_loader(records, shuffle):
    return DataLoader(
        records,
        batch_size=COLES_BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_coles_views,
        **DATALOADER_KWARGS,
    )


def make_ptls_coles_loader(records, shuffle):
    splitter = SampleSlices(
        split_count=COLES_SPLIT_COUNT,
        cnt_min=MIN_SLICE_LEN,
        cnt_max=config["data"]["max_seq_len"],
    )
    dataset = ColesDataset(data=records, splitter=splitter, col_time="event_time")
    return DataLoader(
        dataset,
        batch_size=COLES_BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=dataset.collate_fn,
        **DATALOADER_KWARGS,
    )


def batch_to_device(batch, device):
    return batch.to(device) if hasattr(batch, "to") else batch


def extract_contrastive_labels(batch):
    payload = getattr(batch, "payload", {})
    for key in ("pair_id", "target", "labels", "label", "y"):
        if key in payload:
            labels = payload[key]
            if labels.ndim > 1:
                labels = labels.view(labels.shape[0], -1)[:, 0]
            return labels.long()
    raise KeyError("No contrastive labels found in CoLES batch payload")


def supervised_contrastive_loss(embeddings, labels, temperature=CONTRASTIVE_TEMPERATURE):
    labels = labels.view(-1)
    z = F.normalize(embeddings, dim=-1)
    logits = z @ z.T / temperature
    batch_size = logits.shape[0]
    diag = torch.eye(batch_size, dtype=torch.bool, device=logits.device)
    logits = logits.masked_fill(diag, -1e9)
    positives = labels[:, None].eq(labels[None, :]) & ~diag
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    positive_count = positives.sum(dim=1)
    valid = positive_count > 0
    if not torch.any(valid):
        return embeddings.sum() * 0.0
    loss = -(log_prob * positives.float()).sum(dim=1) / positive_count.clamp_min(1)
    return loss[valid].mean()


def build_coles_loaders():
    if PTLS_COLES_DATASET_AVAILABLE:
        try:
            train_loader = make_ptls_coles_loader(train_records, shuffle=True)
            val_loader = make_ptls_coles_loader(val_records, shuffle=False)
            sample_batch = batch_to_device(next(iter(train_loader)), device)
            _ = extract_contrastive_labels(sample_batch)
            print("Using PTLS ColesDataset + SampleSlices")
            return train_loader, val_loader, "ptls_coles_dataset"
        except Exception as exc:
            print("PTLS ColesDataset did not expose usable labels; falling back:", repr(exc))
    print("Using manual multi-slice CoLES fallback")
    return make_manual_coles_loader(train_records, True), make_manual_coles_loader(val_records, False), "manual_multi_slice"


seed_all(42)

seq_encoder = RnnSeqEncoder(
    trx_encoder=TrxEncoder(
        embeddings=embedding_specs,
        numeric_values={NUM_COL: "identity"},
    ),
    hidden_size=EMBEDDING_SIZE,
    is_reduce_sequence=True,
).to(device)

train_coles_loader, val_coles_loader, COLES_DATASET_MODE = build_coles_loaders()
optimizer = torch.optim.AdamW(seq_encoder.parameters(), lr=COLES_PRETRAIN_LR, weight_decay=1e-5)
scaler = torch.cuda.amp.GradScaler() if USE_AMP else None
best_state = None
best_val_loss = float("inf")

for epoch in range(COLES_EPOCHS):
    seq_encoder.train()
    train_losses = []
    for batch in train_coles_loader:
        batch = batch_to_device(batch, device)
        labels = extract_contrastive_labels(batch).to(device)
        optimizer.zero_grad(set_to_none=True)
        if USE_AMP:
            with torch.autocast(device_type="cuda"):
                emb = seq_encoder(batch)
                loss = supervised_contrastive_loss(emb, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(seq_encoder.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            emb = seq_encoder(batch)
            loss = supervised_contrastive_loss(emb, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(seq_encoder.parameters(), GRAD_CLIP)
            optimizer.step()
        train_losses.append(float(loss.item()))

    seq_encoder.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_coles_loader:
            batch = batch_to_device(batch, device)
            labels = extract_contrastive_labels(batch).to(device)
            emb = seq_encoder(batch)
            val_losses.append(float(supervised_contrastive_loss(emb, labels).item()))
    train_loss = float(np.mean(train_losses)) if train_losses else float("nan")
    val_loss = float(np.mean(val_losses)) if val_losses else float("nan")
    print(f"CoLES epoch={epoch:02d} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in seq_encoder.state_dict().items()}
        torch.save(
            {
                "seq_encoder_state_dict": best_state,
                "val_loss": best_val_loss,
                "dataset_mode": COLES_DATASET_MODE,
                "cutoff_policy": config["forecasting"],
            },
            CKPT_DIR / "coles_seq_encoder.pt",
        )

if best_state is not None:
    seq_encoder.load_state_dict(best_state)
print("CoLES pretraining complete; mode:", COLES_DATASET_MODE)


In [ ]:
# Cell 7 - Supervised CoLES classifier helpers
train_records_labeled = make_ptls_records(df_ptls, splits["train"], include_target=True)
val_records_labeled = make_ptls_records(df_ptls, splits["val"], include_target=True)
test_records_labeled = make_ptls_records(df_ptls, splits["test"], include_target=True)


def collate_labeled_ptls(records):
    labels = torch.tensor([int(item["target"]) for item in records], dtype=torch.long)
    feature_records = [{k: v for k, v in item.items() if k != "target"} for item in records]
    batch = collate_feature_dict(feature_records)
    batch.payload["target"] = labels
    return batch


def make_labeled_loader(entity_ids, shuffle):
    records = make_ptls_records(df_ptls, entity_ids, include_target=True)
    return DataLoader(
        records,
        batch_size=COLES_BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_labeled_ptls,
        **DATALOADER_KWARGS,
    )


val_loader = DataLoader(
    val_records_labeled,
    batch_size=COLES_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_labeled_ptls,
    **DATALOADER_KWARGS,
)
test_loader = DataLoader(
    test_records_labeled,
    batch_size=COLES_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_labeled_ptls,
    **DATALOADER_KWARGS,
)


class CoLESClassifier(nn.Module):
    def __init__(self, seq_encoder, embedding_size, num_classes=2, dropout=0.10):
        super().__init__()
        self.seq_encoder = seq_encoder
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(embedding_size, num_classes),
        )

    def forward(self, batch):
        emb = self.seq_encoder(batch)
        return self.classifier(emb)


def batch_labels(batch):
    labels = batch.payload["target"]
    if labels.ndim > 1:
        labels = labels.view(labels.shape[0], -1)[:, 0]
    return labels.long()


def set_encoder_trainable(model, trainable):
    for param in model.seq_encoder.parameters():
        param.requires_grad = trainable


def build_coles_classifier(mode):
    seq_encoder_copy = copy.deepcopy(seq_encoder)
    model = CoLESClassifier(seq_encoder_copy, EMBEDDING_SIZE).to(device)
    if mode == "frozen_head":
        set_encoder_trainable(model, False)
    elif mode == "full_finetune":
        set_encoder_trainable(model, True)
    else:
        raise ValueError(f"Unknown CoLES downstream mode: {mode}")
    return model


def build_optimizer(model, mode):
    if mode == "frozen_head":
        params = [p for p in model.classifier.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=COLES_HEAD_LR, weight_decay=COLES_WEIGHT_DECAY)
    return torch.optim.AdamW(
        [
            {"params": model.seq_encoder.parameters(), "lr": COLES_ENCODER_LR},
            {"params": model.classifier.parameters(), "lr": COLES_HEAD_LR},
        ],
        weight_decay=COLES_WEIGHT_DECAY,
    )


def evaluate_coles_classifier(model, loader):
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            batch = batch_to_device(batch, device)
            labels = batch_labels(batch)
            logits = model(batch)
            probs = torch.softmax(logits, dim=-1)[:, 1]
            all_probs.append(probs.detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
    y_true = np.concatenate(all_labels)
    y_score = np.concatenate(all_probs)
    metrics = compute_binary_metrics(y_true, y_score)
    pred_df = pd.DataFrame({"target": y_true, "proba": y_score})
    return metrics, pred_df


def train_coles_classifier(mode, train_ids, seed, run_dir):
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    train_loader = make_labeled_loader(train_ids, shuffle=True)
    model = build_coles_classifier(mode)
    optimizer = build_optimizer(model, mode)
    scaler = torch.cuda.amp.GradScaler() if USE_AMP else None

    best_state = None
    best_val_auc = -float("inf")
    patience_counter = 0

    for epoch in range(COLES_FINETUNE_EPOCHS):
        model.train()
        total_loss = 0.0
        total_count = 0
        for batch in train_loader:
            batch = batch_to_device(batch, device)
            labels = batch_labels(batch)
            optimizer.zero_grad(set_to_none=True)
            if USE_AMP:
                with torch.autocast(device_type="cuda"):
                    logits = model(batch)
                    loss = F.cross_entropy(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(batch)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            total_loss += float(loss.item()) * int(labels.shape[0])
            total_count += int(labels.shape[0])

        val_metrics, _ = evaluate_coles_classifier(model, val_loader)
        val_auc = val_metrics["roc_auc"]
        train_loss = total_loss / max(1, total_count)
        print(f"{mode} epoch={epoch:02d} train_loss={train_loss:.4f} val_roc_auc={val_auc:.4f}")
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            run_dir.mkdir(parents=True, exist_ok=True)
            torch.save({"model_state_dict": best_state, "val_metrics": val_metrics}, run_dir / "best.pt")
        else:
            patience_counter += 1
            if patience_counter >= COLES_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_auc

print("Supervised CoLES loaders ready")


In [ ]:
# Cell 8 - Low-label CoLES classifier-head runs and standardized outputs
run_rows = []
prediction_rows = []

for mode in COLES_DOWNSTREAM_MODES:
    pipeline = f"coles_{mode}"
    for fraction in LABEL_FRACTIONS:
        for seed in LABEL_SAMPLING_SEEDS:
            train_ids = sample_labeled_entities(splits["train"], labels_by_entity, fraction, seed)
            run_id = f"{pipeline}_f{int(fraction * 100):03d}_s{seed}"
            run_dir = CKPT_DIR / run_id
            print(f"mode={mode} fraction={fraction:.2f} seed={seed} train_entities={len(train_ids)}")

            model, best_val_auc = train_coles_classifier(mode, train_ids, seed, run_dir)
            test_metrics, pred_df = evaluate_coles_classifier(model, test_loader)
            row = {
                "baseline": "coles_ptls",
                "pipeline": pipeline,
                "mode": mode,
                "fraction": fraction,
                "seed": seed,
                "train_entities": len(train_ids),
                "best_val_roc_auc": float(best_val_auc),
                **test_metrics,
            }
            run_rows.append(row)
            run_dir.mkdir(parents=True, exist_ok=True)
            (run_dir / "test_metrics.json").write_text(json.dumps(row, indent=2, ensure_ascii=False, default=json_default))
            print(row)

            pred_df.insert(0, GROUP_COL, splits["test"][: len(pred_df)])
            pred_df["baseline"] = "coles_ptls"
            pred_df["pipeline"] = pipeline
            pred_df["mode"] = mode
            pred_df["fraction"] = fraction
            pred_df["seed"] = seed
            prediction_rows.append(pred_df)

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

all_runs_df, low_label_agg_df, summary_df, predictions_df, metrics_payload = save_low_label_outputs(
    run_rows,
    prediction_rows,
    baseline_name="coles_ptls",
    output_dir=OUTPUT_DIR,
    extra_artifacts={
        "pretrain_checkpoint": CKPT_DIR / "coles_seq_encoder.pt",
        "coles_dataset_mode": COLES_DATASET_MODE,
    },
)

try:
    display(low_label_agg_df)
except NameError:
    print(low_label_agg_df.to_string(index=False))
